# Build Caravan Dataset — Pipeline Completo
## ANA → preprocessor → caravan_formatter → validação

Este notebook executa o pipeline de ponta a ponta e produz o diretório
`data/processed/Caravan-nc/` que o OpenHydroNet lê durante o treinamento.

### O que este notebook garante antes de treinar:
1. Dados brutos baixados e verificados
2. Pré-processamento aplicado com decisões documentadas
3. Estrutura Caravan criada e validada
4. Resumo do dataset final: período, cobertura, estatísticas

**Por que validar antes de treinar?**  
O framework só levanta `FileNotFoundError` ou `KeyError` durante o primeiro
batch — potencialmente horas depois de iniciar o treino. Este notebook
detecta todos esses problemas em segundos.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, json
sys.path.insert(0, '../../')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from src.data.ana_downloader import download_series, save_raw
from src.data.preprocessor import preprocess
from src.data.caravan_formatter import (
    format_caravan,
    validate_caravan_structure,
    generate_basins_list,
)

plt.style.use('seaborn-v0_8-whitegrid')

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = Path('../..')
RAW_FLOW    = ROOT / 'data/raw/streamflow'
RAW_PRECIP  = ROOT / 'data/raw/precipitation'
PROCESSED   = ROOT / 'data/processed'
CARAVAN_ROOT = PROCESSED / 'Caravan-nc'
FIGDIR      = ROOT / 'reports/figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

STATION = '83500000'
TRAIN_END = '2000-12-31'  # deve bater com train_end_date do YAML

print('Ambiente configurado.')
print(f'Caravan root: {CARAVAN_ROOT.resolve()}')

## 1. Dados Brutos — Download e Verificação

Se os arquivos `.parquet` já existirem, esta seção é um no-op.
Caso contrário, faz o download da série completa (pode levar ~5 min).

In [ ]:
# ── Vazão ─────────────────────────────────────────────────────────────────────
vazao_raw = RAW_FLOW / f'{STATION}_vazao_raw.parquet'
if not vazao_raw.exists():
    print('Baixando vazão (ANA 83500000)...')
    df_v = download_series(STATION, 'vazao', start_year=1940)
    save_raw(df_v, STATION, 'vazao', RAW_FLOW)
    print(f'  → {len(df_v)} dias, {df_v["value"].notna().mean()*100:.1f}% válidos')
else:
    df_v = pd.read_parquet(vazao_raw)
    print(f'Vazão carregada: {df_v.index.min().date()} → {df_v.index.max().date()}')

# ── Cota ──────────────────────────────────────────────────────────────────────
cota_raw = RAW_FLOW / f'{STATION}_cota_raw.parquet'
if not cota_raw.exists():
    print('Baixando cota...')
    df_c = download_series(STATION, 'cota', start_year=1940)
    save_raw(df_c, STATION, 'cota', RAW_FLOW)
else:
    df_c = pd.read_parquet(cota_raw)
    print(f'Cota carregada:  {df_c.index.min().date()} → {df_c.index.max().date()}')

# ── Precipitação (CHIRPS) ─────────────────────────────────────────────────────
chirps_files = sorted(RAW_PRECIP.glob('chirps_*_itajai.nc'))
precip_daily_path = PROCESSED / 'chirps_basin_average_daily.parquet'

if not precip_daily_path.exists():
    if chirps_files:
        print(f'Calculando média areal CHIRPS ({len(chirps_files)} anos)...')
        from src.data.precipitation_downloader import chirps_basin_average
        df_p = chirps_basin_average(RAW_PRECIP)
        PROCESSED.mkdir(exist_ok=True)
        df_p.to_parquet(precip_daily_path)
        print(f'  → {len(df_p)} dias')
    else:
        print('CHIRPS não encontrado. Prosseguindo sem precipitação (pode ser adicionada depois).')
        df_p = None
else:
    df_p = pd.read_parquet(precip_daily_path)
    print(f'Precipitação CHIRPS: {df_p.index.min().date()} → {df_p.index.max().date()}')

print('\nDados brutos OK.')

## 2. Pré-processamento da Vazão

Aplica o pipeline: outliers físicos → interpolação curta → log-transform → Z-score.

**Parâmetro crítico: `train_end`**  
A normalização (Z-score) usa apenas as estatísticas do período de treino.
Se mudarmos o split no YAML, precisamos re-rodar esta célula com
o novo `train_end` para evitar data leakage.

In [ ]:
print(f'Rodando preprocessor (train_end={TRAIN_END})...')
df_processed, norm_stats = preprocess(
    raw_path=vazao_raw,
    variable='vazao',
    out_dir=PROCESSED,
    train_end=TRAIN_END,
    apply_log=True,
)

print('\nEstatísticas de normalização (salvar para inferência!):')
for k, v in norm_stats.items():
    print(f'  {k:15s}: {v}')

# Sanity check: série normalizada deve ter mean ≈ 0 e std ≈ 1 no período de treino
train_norm = df_processed.loc[:TRAIN_END, 'value_norm'].dropna()
print(f'\nVerificação do Z-score no período de treino:')
print(f'  mean = {train_norm.mean():.4f}  (esperado: ~0)')
print(f'  std  = {train_norm.std():.4f}  (esperado: ~1)')

## 3. Visualização: Bruto vs. Processado

Comparação side-by-side para confirmar que o pré-processamento não introduziu
artefatos. Três verificações visuais:
1. Outliers foram removidos (não deve haver picos absurdos)
2. Gaps curtos foram interpolados (não deve haver dentes de serra de NaN)
3. A transformação log comprimiu os picos — distribuição mais simétrica

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)

Q_raw  = df_v['value'].rename('Bruto (m³/s)')
Q_proc = df_processed['value'].rename('Limpo (m³/s)')
Q_log  = df_processed['value_log'].rename('log(1+Q)')
Q_norm = df_processed['value_norm'].rename('Normalizado (Z-score)')
interp = df_processed['interpolated']

# 1. Bruto vs. Limpo
axes[0].plot(Q_raw.index, Q_raw.values, lw=0.4, color='gray', alpha=0.7, label='Bruto')
axes[0].plot(Q_proc.index, Q_proc.values, lw=0.4, color='steelblue', label='Limpo')
if interp.sum() > 0:
    axes[0].scatter(interp[interp].index, Q_proc[interp],
                    c='orange', s=4, zorder=5, label='Interpolado')
axes[0].set_ylabel('Vazão (m³/s)')
axes[0].legend(fontsize=8)

# 2. Log-transform
axes[1].plot(Q_log.index, Q_log.values, lw=0.4, color='seagreen')
axes[1].set_ylabel('log(1 + Q)')

# 3. Z-score normalizado
axes[2].plot(Q_norm.index, Q_norm.values, lw=0.4, color='darkorchid')
axes[2].axhline(0, color='k', lw=0.5, ls='--')
axes[2].axvline(pd.Timestamp(TRAIN_END), color='crimson', lw=1.5, ls='--',
                label=f'Fim treino ({TRAIN_END})')
axes[2].set_ylabel('Normalizado (Z-score)')
axes[2].legend(fontsize=8)

axes[-1].set_xlabel('Data')
fig.suptitle('Pipeline de Pré-processamento — Vazão Itajaí-Açu / Blumenau', fontsize=12)
fig.tight_layout()
fig.savefig(FIGDIR / '16_preprocessing_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Estatísticas de gaps
from src.data.preprocessor import classify_gaps
gaps = classify_gaps(df_processed['value'])
print(f'Gaps residuais após interpolação: {len(gaps)}')
if not gaps.empty:
    print(gaps.sort_values('days', ascending=False).head(10).to_string())

## 4. Geração do Caravan Dataset

Converte os dados locais para o formato esperado pelo OpenHydroNet.
Produz:
```
data/processed/Caravan-nc/
├── timeseries/
│   ├── csv/itajai/itajai_83500000.csv     ← hindcast + targets (load_as_csv: true)
│   └── netcdf/itajai/itajai_83500000.nc   ← referência / modo zarr futuro
└── attributes/itajai/
    ├── attributes_caravan_itajai.csv
    ├── attributes_hydroatlas_itajai.csv
    └── attributes_other_itajai.csv
```

In [ ]:
processed_vazao_path = PROCESSED / f'{STATION}_vazao_processed.parquet'

print('Rodando caravan_formatter...')
paths = format_caravan(
    streamflow_parquet=processed_vazao_path,
    precipitation_parquet=precip_daily_path if (df_p is not None) else None,
    caravan_root=CARAVAN_ROOT,
)

print('\nArquivos gerados:')
for key, path in paths.items():
    size_kb = Path(path).stat().st_size / 1024
    print(f'  {key:20s}: {path.name} ({size_kb:.1f} KB)')

# Gerar arquivo de lista de bacias para o YAML
basins_txt = ROOT / 'configs/training/itajai_basins.txt'
generate_basins_list(CARAVAN_ROOT, out_path=basins_txt)
print(f'\nLista de bacias: {basins_txt}')
print(basins_txt.read_text())

## 5. Validação da Estrutura Caravan

Este é o **gate de qualidade** antes do treinamento.
Todos os checks devem ser `True` antes de rodar `run train`.

In [ ]:
print('Validando estrutura Caravan...')
checks = validate_caravan_structure(CARAVAN_ROOT)

all_ok = all(checks.values())
status_icon = '✓' if all_ok else '✗'

print(f'\nResultado: {status_icon} {"APROVADO" if all_ok else "REPROVADO"}')
print('-' * 40)
for check, result in checks.items():
    icon = '✓' if result else '✗  ← PROBLEMA'
    print(f'  {icon}  {check}')

if not all_ok:
    print('\nCorreções necessárias antes do treinamento:')
    for check, result in checks.items():
        if not result:
            print(f'  - {check}')

## 6. Inspeção do NetCDF

Abrimos o arquivo gerado e verificamos:
- Dimensões e variáveis corretas
- Datas contínuas sem gaps no índice
- Valores dentro dos limites físicos esperados
- Alinhamento temporal entre streamflow e precipitation

In [ ]:
nc_path = CARAVAN_ROOT / 'timeseries/netcdf/itajai/itajai_83500000.nc'
ds = xr.open_dataset(nc_path)

print('=== NetCDF Info ===')
print(ds)

print('\n=== Datas ===' )
dates = pd.DatetimeIndex(ds['date'].values)
print(f'Início:  {dates[0].date()}')
print(f'Fim:     {dates[-1].date()}')
print(f'Total:   {len(dates)} dias')

# Verificar continuidade do índice
gaps_in_index = pd.date_range(dates[0], dates[-1], freq='D').difference(dates)
print(f'Gaps no índice: {len(gaps_in_index)}  (esperado: 0)')

print('\n=== Streamflow ===')
q = ds['streamflow'].to_series()
print(f'  NaN:  {q.isna().sum()} ({q.isna().mean()*100:.1f}%)')
print(f'  min:  {q.min():.1f} m³/s')
print(f'  max:  {q.max():.1f} m³/s  (esperado < 15000)')
print(f'  mean: {q.mean():.1f} m³/s')

if 'total_precipitation' in ds:
    p = ds['total_precipitation'].to_series()
    print('\n=== Precipitação ===')
    print(f'  NaN:  {p.isna().sum()} ({p.isna().mean()*100:.1f}%)')
    print(f'  max:  {p.max():.1f} mm/dia')
    # Verificar cobertura temporal coincidente com streamflow
    both_valid = (~q.isna()) & (~p.isna())
    print(f'  Período com ambos válidos: {both_valid.sum()} dias')

ds.close()

## 7. Resumo Final do Dataset

Tabela de resumo que documenta o estado do dataset antes do treinamento.
Salvar este output como referência junto com os logs de treinamento.

In [ ]:
ds = xr.open_dataset(nc_path)
dates = pd.DatetimeIndex(ds['date'].values)
q = ds['streamflow'].to_series()

# Splits
train_mask = dates <= TRAIN_END
val_mask   = (dates > TRAIN_END) & (dates <= '2010-12-31')
test_mask  = dates > '2010-12-31'

summary = {
    'Período total':     f'{dates[0].date()} → {dates[-1].date()}',
    'Total de dias':     len(dates),
    'Vazão: cobertura total':  f'{q.notna().mean()*100:.1f}%',
    'Vazão: cobertura treino': f'{q[train_mask].notna().mean()*100:.1f}%',
    'Vazão: cobertura val':    f'{q[val_mask].notna().mean()*100:.1f}%',
    'Vazão: cobertura teste':  f'{q[test_mask].notna().mean()*100:.1f}%',
    'Vazão: max histórico':    f'{q.max():.0f} m³/s',
    'Normalização: mean':      f'{norm_stats["mean"]:.4f}',
    'Normalização: std':       f'{norm_stats["std"]:.4f}',
    'Normalização: train_end': norm_stats['train_end'],
    'Caravan: todos checks':   '✓ OK' if all(checks.values()) else '✗ VER ACIMA',
}

if 'total_precipitation' in ds:
    p = ds['total_precipitation'].to_series()
    summary['Precipitação: cobertura total'] = f'{p.notna().mean()*100:.1f}%'

print('=' * 52)
print('  RESUMO DO DATASET — PRONTO PARA TREINAMENTO')
print('=' * 52)
for k, v in summary.items():
    print(f'  {k:<35s} {v}')
print('=' * 52)

# Salvar como JSON
summary_path = PROCESSED / 'dataset_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSalvo: {summary_path}')

ds.close()

## 8. Plot de Cobertura por Split

Visualização temporal mostrando o que o modelo "vê" em cada fase.

In [ ]:
ds = xr.open_dataset(nc_path)
q = ds['streamflow'].to_series()
dates = q.index

fig, ax = plt.subplots(figsize=(16, 4))

TRAIN_END_DT = pd.Timestamp(TRAIN_END)
VAL_END_DT   = pd.Timestamp('2010-12-31')

# Sombrear splits
ax.axvspan(dates[0], TRAIN_END_DT, alpha=0.12, color='steelblue', label='Treino')
ax.axvspan(TRAIN_END_DT, VAL_END_DT, alpha=0.12, color='seagreen', label='Validação')
ax.axvspan(VAL_END_DT, dates[-1], alpha=0.12, color='salmon', label='Teste')

ax.plot(q.index, q.values, lw=0.4, color='k', alpha=0.8)

# Linhas de divisão
for dt, label in [(TRAIN_END_DT, TRAIN_END), (VAL_END_DT, '2010-12-31')]:
    ax.axvline(dt, color='gray', lw=1.2, ls='--')
    ax.text(dt, ax.get_ylim()[1]*0.95 if ax.get_ylim()[1] > 0 else 1000,
            label, ha='center', va='top', fontsize=8, color='gray')

ax.set_ylabel('Vazão (m³/s)')
ax.set_title('Dataset Final — Splits Treino / Validação / Teste', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.tight_layout()
fig.savefig(FIGDIR / '17_dataset_splits.png', dpi=150, bbox_inches='tight')
plt.show()
ds.close()